<a href="https://colab.research.google.com/github/harinireddy2308/Nutriseeker/blob/main/NutriSeeker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install all required libraries
!pip install transformers torch gradio Pillow requests -q

import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries installed!")

✅ All libraries installed!


In [2]:
# Create sample IFCT database
ifct_data = """food_name,calories,protein,carbs,fat,fiber
rice,130,2.7,28.0,0.3,0.4
brown rice,216,4.5,44.8,1.8,3.5
roti,120,3.5,24.0,0.4,2.1
chapati,120,3.5,24.0,0.4,2.1
dal,116,7.6,20.0,0.4,3.8
rajma,144,8.7,26.0,0.5,6.4
biryani,163,5.2,25.0,4.1,1.2
idli,58,1.9,11.0,0.1,0.5
dosa,168,3.4,25.0,6.4,1.2
sambar,42,2.3,7.0,0.6,2.1
paneer,265,18.3,1.2,20.8,0.0
chicken curry,243,28.0,8.0,11.0,1.2
egg,155,13.0,1.1,11.0,0.0
bread,265,9.0,49.0,3.2,2.7
banana,89,1.1,23.0,0.3,2.6
apple,52,0.3,14.0,0.2,2.4
milk,42,3.4,5.0,1.0,0.0
curd,98,11.0,3.4,4.3,0.0
poha,333,7.0,76.0,0.5,2.7
upma,145,3.9,24.0,4.2,2.1
puri,326,7.0,45.0,14.0,2.1
aloo sabzi,98,2.1,18.0,2.8,2.4
palak paneer,187,9.8,8.3,13.0,2.6
butter chicken,164,14.0,5.0,10.0,0.8
naan,317,10.0,55.0,6.0,2.0
pizza,266,11.0,33.0,10.0,2.3
burger,295,17.0,24.0,14.0,1.5
sandwich,250,10.0,30.0,10.0,2.0
soup,72,3.5,10.0,2.0,1.5
salad,65,2.0,10.0,3.0,3.0
pasta,220,8.0,43.0,1.3,2.5
noodles,138,4.5,25.0,2.0,1.5
oats,389,17.0,66.0,7.0,10.0
cornflakes,357,7.0,84.0,0.4,3.0"""

with open("ifct_sample.csv", "w") as f:
    f.write(ifct_data)

print("✅ IFCT database created successfully!")

✅ IFCT database created successfully!


In [3]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

print("⏳ Loading BLIP model... (first time takes 2-3 mins)")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✅ BLIP model loaded! Running on: {device}")

⏳ Loading BLIP model... (first time takes 2-3 mins)


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

✅ BLIP model loaded! Running on: cuda


In [4]:
import pandas as pd
import requests

df = pd.read_csv("ifct_sample.csv")

FOOD_KEYWORD_MAP = {
    "rice": "rice",
    "biryani": "biryani",
    "roti": "roti",
    "chapati": "roti",
    "bread": "bread",
    "naan": "naan",
    "dal": "dal",
    "lentil": "dal",
    "curry": "chicken curry",
    "chicken": "chicken curry",
    "egg": "egg",
    "omelette": "egg",
    "dosa": "dosa",
    "idli": "idli",
    "upma": "upma",
    "poha": "poha",
    "paneer": "paneer",
    "potato": "aloo sabzi",
    "aloo": "aloo sabzi",
    "banana": "banana",
    "apple": "apple",
    "milk": "milk",
    "curd": "curd",
    "yogurt": "curd",
    "pasta": "pasta",
    "noodle": "noodles",
    "puri": "puri",
    "pizza": "pizza",
    "burger": "burger",
    "sandwich": "sandwich",
    "soup": "soup",
    "salad": "salad",
    "oat": "oats",
    "cornflake": "cornflakes",
    "rajma": "rajma",
    "sambar": "sambar",
    "palak": "palak paneer",
    "butter": "butter chicken",
}

def search_ifct(food_name):
    food_name = food_name.lower().strip()

    # Step 1 — keyword map
    for keyword, mapped_name in FOOD_KEYWORD_MAP.items():
        if keyword in food_name:
            result = df[df['food_name'].str.lower() == mapped_name.lower()]
            if not result.empty:
                print(f"Keyword match: '{keyword}' → '{mapped_name}'")
                data = result.iloc[0]
                return {
                    "source": "IFCT",
                    "food": data['food_name'],
                    "calories": data['calories'],
                    "protein": data['protein'],
                    "carbs": data['carbs'],
                    "fat": data['fat'],
                    "fiber": data['fiber']
                }

    # Step 2 — partial match
    result = df[df['food_name'].str.contains(food_name, case=False, na=False)]
    if not result.empty:
        data = result.iloc[0]
        return {
            "source": "IFCT",
            "food": data['food_name'],
            "calories": data['calories'],
            "protein": data['protein'],
            "carbs": data['carbs'],
            "fat": data['fat'],
            "fiber": data['fiber']
        }

    return None

def search_usda(food_name):
    try:
        api_key = "DEMO_KEY"
        url = "https://api.nal.usda.gov/fdc/v1/foods/search"
        params = {"query": food_name, "api_key": api_key, "pageSize": 1}
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        if data.get('foods'):
            food = data['foods'][0]
            nutrients = {n['nutrientName']: n['value'] for n in food.get('foodNutrients', [])}
            return {
                "source": "USDA",
                "food": food['description'],
                "calories": nutrients.get('Energy', 0),
                "protein": nutrients.get('Protein', 0),
                "carbs": nutrients.get('Carbohydrate, by difference', 0),
                "fat": nutrients.get('Total lipid (fat)', 0),
                "fiber": nutrients.get('Fiber, total dietary', 0)
            }
    except:
        pass
    return None

def get_nutrition(food_name):
    result = search_ifct(food_name)
    if result:
        print("✅ Found in IFCT!")
        return result
    print("⚠️ Not in IFCT, trying USDA...")
    result = search_usda(food_name)
    if result:
        print("✅ Found in USDA!")
        return result
    return None

print("✅ Nutrition lookup ready!")

✅ Nutrition lookup ready!


In [7]:
def identify_food(image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)

    # ✅ NO text parameter — captioning model needs image only
    inputs = processor(images=image, return_tensors="pt").to(device)

    output = model.generate(**inputs, max_new_tokens=50)
    full_description = processor.decode(output[0], skip_special_tokens=True)

    # Clean BLIP output
    food_name = full_description.lower()
    remove_phrases = [
        "a plate of", "a bowl of", "a dish of", "a serving of",
        "a photo of", "a picture of", "an image of", "i think this is",
        "this is", "the image shows", "there is"
    ]
    for phrase in remove_phrases:
        food_name = food_name.replace(phrase, "")

    for separator in [" and ", " with ", ",", " on a", " on the"]:
        food_name = food_name.split(separator)[0]

    food_name = food_name.strip()
    print(f"BLIP raw output   : {full_description}")
    print(f"Cleaned food name : {food_name}")

    return food_name, full_description

print("✅ Food identifier ready!")

✅ Food identifier ready!


In [8]:
import gradio as gr

def analyze_meal(image):
    try:
        if image is None:
            return "❌ Please upload an image!"

        # Step 1 — Detect food
        food_name, full_description = identify_food(image)

        # Step 2 — Lookup nutrition
        nutrition = get_nutrition(food_name)

        # Step 3 — Display results
        if nutrition:
            result = f"""
🍽️  MEAL ANALYSIS RESULTS
{'='*40}
📌 BLIP Detected  : {full_description}
🔍 Matched Food   : {nutrition['food']}
📚 Data Source    : {nutrition['source']}

{'='*40}
   NUTRITION INFO (per 100g)
{'='*40}
🔥 Calories  :  {nutrition['calories']} kcal
💪 Protein   :  {nutrition['protein']} g
🍞 Carbs     :  {nutrition['carbs']} g
🧈 Fat       :  {nutrition['fat']} g
🌾 Fiber     :  {nutrition['fiber']} g
{'='*40}
            """
        else:
            result = f"""
🍽️  MEAL ANALYSIS RESULTS
{'='*40}
📌 BLIP Detected : {full_description}
❌ Food not found in IFCT or USDA

💡 Tips to improve detection:
  • Use a clearer, well-lit photo
  • Make sure one food item is visible
  • Try a closer shot of the dish
{'='*40}
            """
        return result

    except Exception as e:
        return f"❌ Error: {str(e)}\nPlease re-run all cells from top (Cell 1 to 6)."

# Launch Gradio
try:
    demo.close()
except:
    pass

demo = gr.Interface(
    fn=analyze_meal,
    inputs=gr.Image(type="pil", label="📸 Upload Your Meal Photo"),
    outputs=gr.Textbox(label="📊 Nutrition Results", lines=18),
    title="🍽️ NutriSeeker — AI Meal Nutrition Analyzer",
    description="Upload a food photo to get nutritional information powered by BLIP + IFCT/USDA",
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cad012354da926fe4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
